## TOGSim Log Analysis

In [ ]:
import os
os.environ["TORCHSIM_TARGET"] = os.path.join(os.getcwd(), "togsim_configs", "togsim_config_timing_only.yml")
os.environ["TORCHSIM_LOG_PATH"] = os.path.join(os.getcwd(), "info_results")
os.environ["TORCHSIM_DUMP_PATH"] = os.path.join(os.getcwd(), "outputs")
os.environ["TORCHSIM_NPUWATTCH"] = "1"  # also write npuwattch_<run-id>.json for NPUWattch
import glob
import torch

def latest_log():
    """The newest TOGSim log under TORCHSIM_LOG_PATH."""
    return max(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")), key=os.path.getmtime)

_seen = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log")))

def new_logs():
    """The TOGSim logs written since the last call, oldest first: one per simulation, two for a
    resident mm (its operand's copy-in, then its body), whose cycles add up."""
    logs = set(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "*.log"))) - _seen
    _seen.update(logs)
    return sorted(logs, key=os.path.getmtime)

### log level info

In [ ]:
device = torch.device("npu:0")

input = torch.randn(1024, 1024, dtype=torch.bfloat16).to(device=device)
weight = torch.randn(1024, 1024, dtype=torch.bfloat16).to(device=device)

opt_fn = torch.compile(dynamic=False)(torch.matmul)
npu_outut = opt_fn(input, weight)

In [ ]:
!cat {latest_log()}

### Output files
Each simulation writes a `.log`, a `.trace` and a `.json`. With `TORCHSIM_NPUWATTCH=1` the run also writes one `npuwattch_<run-id>.json`, rewritten after each launch: an entry per kernel with the counters NPUWattch's power model reads.

In [ ]:
!ls {os.environ["TORCHSIM_LOG_PATH"]}

In [ ]:
import json
npuwattch = max(glob.glob(os.path.join(os.environ["TORCHSIM_LOG_PATH"], "npuwattch_*.json")), key=os.path.getmtime)
doc = json.load(open(npuwattch))
print(doc["schema"], doc["schema_version"])
for k in doc["kernels"]:
    print(k["id"], k["name"], k["operation"], k["dtype"], "total_cycles:", k["total_cycles"], "useful_macs:", k["useful_macs"])